# Embeddings

In [1]:
!pip install datasets staticvectors scikit-learn nltk -q

## Loading IMDB dataset

In [2]:
import pandas as pd
from datasets import load_dataset

In [3]:
dataset = load_dataset('imdb')

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
train = dataset['train'].shuffle(seed=42).select(range(5000))
test = dataset['test'].shuffle(seed=42).select(range(1000))

In [6]:
train = pd.DataFrame(train)
test = pd.DataFrame(test)

In [7]:
train.head()

,text,label
0,There is no relation at all between Fortier an...,1
1,This movie is a great. The plot is very true t...,1
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0
3,In the process of trying to establish the audi...,1
4,"Yeh, I know -- you're quivering with excitemen...",0


## Preprocessing

In [8]:
import nltk, re

In [9]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    return tokens

In [10]:
train['tokens'] = train['text'].apply(preprocess)
test['tokens'] = test['text'].apply(preprocess)

In [11]:
train.head()

,text,label,tokens
0,There is no relation at all between Fortier an...,1,"[there, is, no, relation, at, all, between, fo..."
1,This movie is a great. The plot is very true t...,1,"[this, movie, is, a, great, the, plot, is, ver..."
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0,"[george, p, cosmatos, rambo, first, blood, par..."
3,In the process of trying to establish the audi...,1,"[in, the, process, of, trying, to, establish, ..."
4,"Yeh, I know -- you're quivering with excitemen...",0,"[yeh, i, know, youre, quivering, with, excitem..."


## Using GloVe

In [12]:
from staticvectors import StaticVectors

In [13]:
model = StaticVectors('neuml/glove-6B')

In [14]:
text = ['this', 'is', 'the', 'best', 'scifi', 'that', 'i', 'have', 'seen', 'in', 'my', 'years', 'of', 'watching', 'scifi', 'i', 'also', 'believe', 'that', 'dark', 'angel', 'will', 'become', 'a', 'cult', 'favorite', 'the', 'action', 'is', 'great', 'but', 'jessica', 'alba', 'is', 'the', 'best', 'and', 'most', 'gorgeous', 'star', 'on', 'tv', 'today']

In [15]:
len(text)

43

In [16]:
testing = model.embeddings(text)

In [17]:
testing.shape

(43, 300)

## Sentence Embedding

In [18]:
import numpy as np

In [19]:
def sentence_embedding(tokens, model, dim=300):
    try:
        vectors = model.embeddings(tokens)
        return np.sum(vectors, axis=0)
    except:
        return np.zeros(dim)

In [20]:
vectors = sentence_embedding(text, model)
vectors.shape

(300,)

## Training and testing datasets

In [21]:
X_train = np.array([sentence_embedding(tokens, model, 300) for tokens in train['tokens']])
y_train = train['label'].values

In [22]:
print(f'{X_train.shape} {y_train.shape}')

(5000, 300) (5000,)


In [23]:
X_test = np.array([sentence_embedding(tokens, model, 300) for tokens in test['tokens']])
y_test = test['label'].values

In [24]:
print(f'{X_test.shape} {y_test.shape}')

(1000, 300) (1000,)


## Classifier

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [26]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [27]:
y_pred = clf.predict(X_test)

In [28]:
y_pred[:10]

array([1, 1, 0, 0, 0, 1, 1, 0, 0, 1])

## Evaluating

In [29]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.74      0.75      0.74       512
           1       0.73      0.73      0.73       488

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000

